# ROI Cropping with SAM 2 &mdash; the **C** operator

Cell-by-cell reproduction of the eardrum ROI zoom-crop used to build the *ROI
content split* that `../2_wct2_augmentation/augment.ipynb` consumes (with `CONTENT_VARIANT="roi"`). Run top-to-bottom with
the **`py-sam2`** kernel.

**Method**
1. SAM 2 segments the eardrum from a single center-point prompt (best of the
   multi-mask outputs).
2. Square the eardrum bounding box (`max(w, h)`).
3. Extend every side by `BUFFER` (0.20) to keep surrounding ear-canal context.
4. **Shrink** the crop if needed so the visible ROI stays inside both the image
   and the circular endoscope field of view &mdash; no black padding, no curved
   FOV edge.
5. Resize to 256&times;256 and apply an inscribed circular mask.

**Manual review (part of the paper procedure).** Every crop was inspected by
hand; when the eardrum was not clearly visible it was re-cropped manually. Below,
review the crops, add overrides to the `MANUAL` dict, and re-run.

The heavy lifting lives in `roi_crop_sam2.py` (the same code the CLI uses), so
this notebook only orchestrates and visualizes.

> Ravi et al. (2024). *SAM 2: Segment Anything in Images and Videos.*


## 1 &nbsp; Configuration

`MODE` selects the input/output paths. `demo` crops the few raw frames under
`../demo_data/`; `real` reads your own raw endoscope frames and writes the ROI
content split that `../2_wct2_augmentation/augment.ipynb` (`CONTENT_VARIANT="roi"`)
consumes.

SAM 2 must be installed and its checkpoint downloaded first (see
`requirements-sam2.txt`); set `SAM2_CKPT` to the checkpoint path.

In [ ]:
import os, sys, glob, csv
import numpy as np, cv2, torch
import matplotlib.pyplot as plt

sys.path.insert(0, ".")            # local module: roi_crop_sam2.py (single source of truth)
import roi_crop_sam2 as R

MODE = "demo"                      # "demo" or "real"

# SAM 2 model. Install SAM 2 and download the checkpoint first (see
# requirements-sam2.txt); set SAM2_CKPT to its path.
SAM2_CKPT = os.environ.get(
    "SAM2_CKPT", os.path.expanduser("~/sam2_checkpoints/sam2.1_hiera_large.pt"))
SAM2_CFG  = "configs/sam2.1/sam2.1_hiera_l.yaml"

if MODE == "demo":
    INPUT_DIR  = "../demo_data/augmentation/content_raw"   # a few raw endoscope frames
    OUTPUT_DIR = "./demo_output_roi"                       # cropped ROI written here
    N          = 12
elif MODE == "real":
    # Raw endoscope frames in -> ROI content split out. The ROI split is what
    # augment.ipynb (CONTENT_VARIANT="roi") reads.
    DATA_ROOT  = os.environ.get("OTOSCOPE_DATA_ROOT", "../real_data")
    INPUT_DIR  = f"{DATA_ROOT}/f2_raw/train"               # raw content split (uncropped frames)
    OUTPUT_DIR = f"{DATA_ROOT}/f2_roi/train"               # ROI content split
    N          = 20                                        # preview count; Section 8 loops over all
else:
    raise ValueError("MODE must be 'demo' or 'real'")

# ---- parameters (paper run) ----
BUFFER     = 0.20
SIZE       = 256
FOV_MARGIN = 0.98                  # <1 hides the curved FOV edge
SUFFIX     = "_roi"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"MODE={MODE}  device={device}")
print("INPUT_DIR :", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 2 &nbsp; Load SAM 2

In [ ]:
R.init_predictor(SAM2_CKPT, SAM2_CFG, device)
print("SAM2 predictor ready")

## 3 &nbsp; List inputs

In [ ]:
EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
files = sorted(p for p in glob.glob(os.path.join(INPUT_DIR, "*"))
               if os.path.splitext(p)[1].lower() in EXTS)
sample = files[:N]
print(f"{len(files)} images total; previewing {len(sample)}")

## 4 &nbsp; Manual overrides

Leave empty for the first pass. After reviewing (Section&nbsp;5) add an entry for
any image whose eardrum is not clearly cropped, then re-run Sections&nbsp;5&ndash;7.

- `"point": [x, y]` &mdash; a better SAM foreground point in raw pixels.
- `"box": [cx, cy, half]` &mdash; an explicit crop window in raw pixels (bypasses SAM).

In [ ]:
MANUAL = {
    # "0007_NORMAL_0007":  {"point": [250, 260]},       # a better SAM point (raw px)
    # "0007_NORMAL_0007":  {"box":   [256, 256, 150]},   # explicit crop window (raw px)
}

## 5 &nbsp; Process one image + review grid

In [ ]:
def process(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    img = R.imread_rgb(path)
    ov = MANUAL.get(stem, {})
    if "box" in ov:                                    # full manual window
        cx, cy, half = ov["box"]
        crop = R.resize_sq(R.crop_roi_circular(img, cx, cy, half), SIZE)
        return stem, img, None, crop, dict(cx=cx, cy=cy, half=half, half_req=half, clamped=0, fov_R="")
    crop, mask, meta = R.make_roi_crop(
        img, "box", BUFFER, 0.5, SIZE,
        point_xy=tuple(ov["point"]) if "point" in ov else None,
        fit_fov=True, fov_margin=FOV_MARGIN, return_extra=True)
    return stem, img, mask, crop, meta


def review(paths, cols=5):
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 2.9))
    for ax in np.array(axes).reshape(-1):
        ax.axis("off")
    for ax, path in zip(np.array(axes).reshape(-1), paths):
        stem, img, mask, crop, meta = process(path)
        ax.imshow(crop)
        tag = " [MANUAL]" if stem in MANUAL else (" [fit]" if meta["clamped"] else "")
        ax.set_title(f"{stem}{tag}", fontsize=7)
    plt.tight_layout(); plt.show()

review(sample)

## 6 &nbsp; Inspect a single image

Pass an index (0-based, into `sample`) or a file stem. Use this to decide a
manual `point`/`box`, then add it to `MANUAL` above and re-run.

In [ ]:
def inspect(which):
    path = sample[which] if isinstance(which, int) else \
           next(p for p in files if os.path.splitext(os.path.basename(p))[0] == which)
    stem, img, mask, crop, meta = process(path)
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(img); ax[0].set_title(f"{stem} - raw"); ax[0].axis("off")
    ovl = img.copy()
    if mask is not None:
        ovl[mask] = (0.5 * ovl[mask] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
        x, y, w, h = R.eardrum_bbox(mask)
        cv2.rectangle(ovl, (x, y), (x + w, y + h), (255, 200, 0), 2)
    cv2.circle(ovl, (int(meta["cx"]), int(meta["cy"])), int(meta["half"]), (255, 0, 0), 2)
    ax[1].imshow(ovl); ax[1].set_title("SAM2 mask + eardrum bbox + ROI circle"); ax[1].axis("off")
    ax[2].imshow(crop); ax[2].set_title(f"crop 256 (clamped={meta['clamped']})"); ax[2].axis("off")
    plt.tight_layout(); plt.show()
    print(meta)

inspect(0)

## 7 &nbsp; Save crops + manifest

In [ ]:
assert os.path.abspath(OUTPUT_DIR) != os.path.abspath(INPUT_DIR), "OUTPUT_DIR must differ from INPUT_DIR"
os.makedirs(OUTPUT_DIR, exist_ok=True)
rows = []
for path in sample:                                  # raise N (Section 1) or loop over `files` for the full set
    stem, img, mask, crop, meta = process(path)
    R.imwrite_rgb(os.path.join(OUTPUT_DIR, f"{stem}{SUFFIX}.png"), crop)
    rows.append([stem, round(meta["cx"], 1), round(meta["cy"], 1), round(meta["half"], 1),
                 round(meta["half_req"], 1), meta["clamped"], meta["fov_R"], int(stem in MANUAL)])
with open(os.path.join(OUTPUT_DIR, "roi_crop_manifest.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["stem", "cx", "cy", "half", "half_requested", "clamped_for_fov", "fov_radius", "manual"])
    w.writerows(rows)
print(f"saved {len(rows)} crops -> {OUTPUT_DIR}  (manual: {sum(r[-1] for r in rows)}, fov-fit: {sum(r[5] for r in rows)})")

## 8 &nbsp; Full dataset / calibration

To crop the whole set, raise `N` above (or loop over `files` in Section&nbsp;7),
or run the CLI:

```bash
python roi_crop_sam2.py --input <frames_dir> --output <roi_split_dir> \
    --crop-mode box --buffer 0.20 --size 256 \
    --manifest <roi_split_dir>/roi_crop_manifest.csv
```

The buffer can be re-derived from an existing crop set:

```bash
python roi_crop_sam2.py --calibrate <existing_roi_crops> [--calibrate-raw <raw_frames>]
```
